# LAB 1 — Construção e Validação do Dataset de Avaliação com Ground Truth

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Avaliação de Pipelines RAG com RAGAS**

---

## Objetivo

Construir e validar um dataset de **50 pares de perguntas e respostas jurídicas** com seus respectivos contextos recuperados e ground truths, preparando o conjunto de avaliação para uso nas métricas RAGAS nos laboratórios subsequentes.

## Técnica Avaliada

- **#T00 — Preparação do Dataset de Avaliação**  
- Pipeline Naive RAG (Aula 4) como gerador de respostas e contextos

## Pré-requisitos

| Componente | Endereço | Observação |
|---|---|---|
| vLLM (Llama 3.1 8B Instruct) | `localhost:8000/v1` | API compatível OpenAI |
| OpenSearch 3.x | `localhost:9200` | Índice `juridico_baseline_aula4` |
| LangFuse | `localhost:3000` | Observabilidade |
| Dataset de entrada | `../datasets/corpus_avaliacao_aula5.json` | 50 pares QA jurídicos |

## Referência Normativa

Conforme **ABNT NBR 6023:2018** (Referências bibliográficas) e **ABNT NBR 10520:2023** (Citações).  
Modelo de linguagem: Meta. **Llama 3.1**. 2024. Disponível em: <https://llama.meta.com>. Acesso em: abr. 2026.  
Framework de avaliação: ES-JOLLY, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. 2023.

---
## Célula 1 — Instalação de Dependências

In [ ]:
# Instalação das dependências necessárias para o laboratório
# Compatível com Python 3.11+ e Google Colab T4
!pip install -q ragas datasets langchain langchain-openai sentence-transformers opensearch-py langfuse pandas

print("Dependências instaladas com sucesso.")

---
## Célula 2 — Configuração de Variáveis de Ambiente

In [ ]:
import os

# ── OpenSearch ─────────────────────────────────────────────────────────────
os.environ["OPENSEARCH_HOST"] = os.getenv("OPENSEARCH_HOST", "localhost")
os.environ["OPENSEARCH_PORT"] = os.getenv("OPENSEARCH_PORT", "9200")
os.environ["OPENSEARCH_USER"] = os.getenv("OPENSEARCH_USER", "admin")
os.environ["OPENSEARCH_PASS"] = os.getenv("OPENSEARCH_PASS", "admin")

# ── vLLM ───────────────────────────────────────────────────────────────────
os.environ["VLLM_HOST"]     = os.getenv("VLLM_HOST",     "localhost")
os.environ["VLLM_PORT"]     = os.getenv("VLLM_PORT",     "8000")
os.environ["VLLM_BASE_URL"] = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
os.environ["VLLM_MODEL"]    = os.getenv("VLLM_MODEL",    "meta-llama/Meta-Llama-3.1-8B-Instruct")

# ── LangFuse ───────────────────────────────────────────────────────────────
os.environ["LANGFUSE_HOST"]       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-SUBSTITUA")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-SUBSTITUA")

# ── Modelo de Embedding ────────────────────────────────────────────────────
os.environ["EMBEDDING_MODEL"] = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Índice OpenSearch ──────────────────────────────────────────────────────
OPENSEARCH_INDEX = "juridico_baseline_aula4"
DATASET_PATH     = "../datasets/corpus_avaliacao_aula5.json"
K_DOCS           = 5  # número de chunks recuperados por consulta

# Leitura das variáveis para uso nos clientes
OPENSEARCH_HOST = os.environ["OPENSEARCH_HOST"]
OPENSEARCH_PORT = int(os.environ["OPENSEARCH_PORT"])
OPENSEARCH_USER = os.environ["OPENSEARCH_USER"]
OPENSEARCH_PASS = os.environ["OPENSEARCH_PASS"]
VLLM_BASE_URL   = os.environ["VLLM_BASE_URL"]
VLLM_MODEL      = os.environ["VLLM_MODEL"]
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]

print("Variáveis de ambiente configuradas.")
print(f"  OpenSearch : {OPENSEARCH_HOST}:{OPENSEARCH_PORT}")
print(f"  vLLM       : {VLLM_BASE_URL}")
print(f"  Modelo     : {VLLM_MODEL}")
print(f"  Embedding  : {EMBEDDING_MODEL}")
print(f"  LangFuse   : {os.environ['LANGFUSE_HOST']}")

---
## Célula 3 — Inicialização das Conexões e Testes de Conectividade

In [ ]:
from opensearchpy import OpenSearch, ConnectionError as OSConnectionError
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

# ── Cliente OpenSearch ─────────────────────────────────────────────────────
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
    timeout=30,
)

# Teste de conectividade — OpenSearch
OPENSEARCH_OK = False
try:
    info = os_client.info()
    OPENSEARCH_OK = True
    print(f"[OK] OpenSearch conectado — versão: {info['version']['number']}")
    # Verifica existência do índice de baseline
    indice_existe = os_client.indices.exists(index=OPENSEARCH_INDEX)
    if indice_existe:
        contagem = os_client.count(index=OPENSEARCH_INDEX)["count"]
        print(f"[OK] Índice '{OPENSEARCH_INDEX}' encontrado — {contagem} documentos")
    else:
        print(f"[AVISO] Índice '{OPENSEARCH_INDEX}' não encontrado. Será usado fallback de contexto vazio.")
        OPENSEARCH_OK = False
except Exception as exc:
    print(f"[AVISO] OpenSearch não disponível: {exc}")
    print("  Será usado fallback: contexto vazio nas respostas.")

# ── Cliente vLLM via LangChain ─────────────────────────────────────────────
llm = ChatOpenAI(
    model=VLLM_MODEL,
    base_url=VLLM_BASE_URL,
    api_key="dummy",           # vLLM local não requer chave real
    temperature=0.3,
    max_tokens=512,
)

VLLM_OK = False
try:
    resposta_teste = llm.invoke("Responda apenas: OK")
    VLLM_OK = True
    print(f"[OK] vLLM conectado — resposta de teste: {resposta_teste.content.strip()[:50]}")
except Exception as exc:
    print(f"[AVISO] vLLM não disponível: {exc}")

# ── Modelo de Embedding BGE-M3 ─────────────────────────────────────────────
print(f"\nCarregando modelo de embedding: {EMBEDDING_MODEL} ...")
encoder = SentenceTransformer(EMBEDDING_MODEL)
print(f"[OK] SentenceTransformer carregado — dimensão: {encoder.get_sentence_embedding_dimension()}")

# Asserts de saúde mínimos
assert encoder is not None, "Falha ao carregar o encoder de embeddings."
print("\nTodos os asserts de conectividade passaram.")

---
## Célula 4 — Carregamento do Dataset e Análise Estatística

In [ ]:
import json
import pandas as pd

# Carrega o arquivo de 50 pares QA jurídicos
try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        corpus = json.load(f)
    print(f"[OK] Dataset carregado: {len(corpus)} pares")
except FileNotFoundError:
    print(f"[AVISO] Arquivo '{DATASET_PATH}' não encontrado. Criando dataset sintético de demonstração...")
    # Dataset de demonstração para execução em modo offline
    tipos_juridicos = [
        "processual_penal", "ambiental", "constitucional",
        "administrativo", "civil", "trabalhista",
        "seguranca_publica", "direitos_humanos"
    ]
    dificuldades = ["facil", "medio", "dificil"]
    corpus = []
    for i in range(50):
        tipo = tipos_juridicos[i % len(tipos_juridicos)]
        dif  = dificuldades[i % len(dificuldades)]
        corpus.append({
            "id": f"q{i+1:03d}",
            "question": f"Questão jurídica de demonstração número {i+1} sobre {tipo.replace('_', ' ')}?",
            "ground_truth": f"Resposta de referência para a questão {i+1}, abordando aspectos de {tipo.replace('_', ' ')} com nível de dificuldade {dif}.",
            "tipo": tipo,
            "dificuldade": dif
        })
    print(f"[OK] Dataset sintético criado: {len(corpus)} pares")

# Converte para DataFrame para análise estatística
df = pd.DataFrame(corpus)

print("\n" + "="*60)
print("ESTATÍSTICAS DO DATASET")
print("="*60)
print(f"\nTotal de pares QA : {len(df)}")
print(f"Colunas presentes : {list(df.columns)}")

print("\n--- Distribuição por Tipo Jurídico ---")
dist_tipo = df["tipo"].value_counts()
print(dist_tipo.to_string())

print("\n--- Distribuição por Dificuldade ---")
dist_dificuldade = df["dificuldade"].value_counts()
print(dist_dificuldade.to_string())

print("\n--- Exemplo de 2 pares do dataset ---")
for _, row in df.head(2).iterrows():
    print(f"  ID          : {row['id']}")
    print(f"  Tipo        : {row['tipo']}")
    print(f"  Dificuldade : {row['dificuldade']}")
    print(f"  Pergunta    : {row['question'][:100]}...")
    print(f"  Ground Truth: {row['ground_truth'][:100]}...")
    print()

---
## Célula 5 — Definição da Função `gerar_resposta_pipeline`

In [ ]:
from typing import Optional


def gerar_embedding(texto: str) -> list[float]:
    """Gera embedding denso para um texto usando o modelo BGE-M3."""
    vetor = encoder.encode(texto, normalize_embeddings=True)
    return vetor.tolist()


def buscar_chunks_opensearch(query: str, k: int = 5) -> list[str]:
    """
    Recupera os top-k chunks mais relevantes do OpenSearch via busca kNN.

    Parâmetros
    ----------
    query : str
        Texto da pergunta do usuário.
    k : int
        Número de chunks a recuperar (padrão: 5).

    Retorna
    -------
    list[str]
        Lista de trechos de texto recuperados.
    """
    vetor_query = gerar_embedding(query)
    corpo_busca = {
        "size": k,
        "_source": ["texto", "content", "chunk_text", "page_content"],
        "query": {
            "knn": {
                "embedding": {          # campo vetorial padrão do LAB5/Aula4
                    "vector": vetor_query,
                    "k": k
                }
            }
        }
    }
    resposta = os_client.search(index=OPENSEARCH_INDEX, body=corpo_busca)
    chunks = []
    for hit in resposta["hits"]["hits"]:
        fonte = hit["_source"]
        # Tenta os campos mais comuns de texto
        texto = (
            fonte.get("texto")
            or fonte.get("content")
            or fonte.get("chunk_text")
            or fonte.get("page_content")
            or ""
        )
        if texto:
            chunks.append(texto)
    return chunks


def montar_prompt(question: str, contextos: list[str]) -> str:
    """Monta o prompt RAG com contexto jurídico e pergunta do usuário."""
    bloco_contexto = "\n\n".join(
        f"[Trecho {i+1}]\n{ctx}" for i, ctx in enumerate(contextos)
    ) if contextos else "Nenhum contexto disponível no momento."

    return (
        "Você é um assistente jurídico especializado em Direito Brasileiro e Segurança Pública. "
        "Responda à pergunta a seguir com base exclusivamente nos trechos de legislação, "
        "doutrina ou jurisprudência fornecidos abaixo. Seja preciso, cite os dispositivos "
        "legais quando relevante e redija em português formal.\n\n"
        f"CONTEXTO:\n{bloco_contexto}\n\n"
        f"PERGUNTA:\n{question}\n\n"
        "RESPOSTA:"
    )


def gerar_resposta_pipeline(
    question: str,
    ground_truth: str,
    k: int = 5
) -> dict:
    """
    Pipeline completo Naive RAG para um par QA.

    Etapas:
      (a) Gera embedding da query com BGE-M3
      (b) Busca top-k chunks no OpenSearch via kNN
      (c) Monta prompt com contexto + pergunta
      (d) Chama vLLM para gerar resposta
      (e) Retorna dict com os 4 campos RAGAS

    Parâmetros
    ----------
    question    : str — pergunta jurídica
    ground_truth: str — resposta de referência
    k           : int — número de chunks a recuperar

    Retorna
    -------
    dict com campos: question, answer, contexts, ground_truth
    """
    # (b) Recuperação de contextos
    contextos: list[str] = []
    if OPENSEARCH_OK:
        try:
            contextos = buscar_chunks_opensearch(question, k=k)
        except Exception as exc:
            # Fallback: contexto vazio se OpenSearch não disponível
            contextos = []

    # (c) Montagem do prompt
    prompt = montar_prompt(question, contextos)

    # (d) Geração de resposta via vLLM
    answer = ""
    if VLLM_OK:
        try:
            resp = llm.invoke(prompt)
            answer = resp.content.strip()
        except Exception as exc:
            answer = f"[Erro na geração: {exc}]"
    else:
        answer = "[vLLM não disponível — resposta não gerada]"

    # (e) Retorno estruturado no formato RAGAS
    return {
        "question":     question,
        "answer":       answer,
        "contexts":     contextos if contextos else ["Contexto indisponível."],
        "ground_truth": ground_truth,
    }


print("Função 'gerar_resposta_pipeline' definida com sucesso.")

---
## Célula 6 — Processamento dos 50 Pares (Loop com tqdm)

In [ ]:
from tqdm.notebook import tqdm

dataset_completo: list[dict] = []
erros: list[dict] = []

print(f"Processando {len(corpus)} pares QA com pipeline Naive RAG...")
print(f"  Modelo de embedding : {EMBEDDING_MODEL}")
print(f"  Modelo de geração   : {VLLM_MODEL}")
print(f"  Chunks recuperados  : k={K_DOCS}")
print()

for par in tqdm(corpus, desc="Gerando respostas", unit="par"):
    try:
        resultado = gerar_resposta_pipeline(
            question=par["question"],
            ground_truth=par["ground_truth"],
            k=K_DOCS,
        )
        # Preserva metadados originais
        resultado["id"]          = par.get("id", "")
        resultado["tipo"]        = par.get("tipo", "")
        resultado["dificuldade"] = par.get("dificuldade", "")
        dataset_completo.append(resultado)

    except Exception as exc:
        # Registra erro mas continua o loop
        erros.append({"id": par.get("id"), "erro": str(exc)})
        # Adiciona par com contexto vazio como fallback
        dataset_completo.append({
            "id":           par.get("id", ""),
            "question":     par["question"],
            "answer":       f"[Erro: {exc}]",
            "contexts":     ["Contexto indisponível devido a erro."],
            "ground_truth": par["ground_truth"],
            "tipo":         par.get("tipo", ""),
            "dificuldade":  par.get("dificuldade", ""),
        })

print(f"\nProcessamento concluído.")
print(f"  Pares gerados com sucesso : {len(dataset_completo) - len(erros)}")
print(f"  Pares com erro (fallback) : {len(erros)}")
if erros:
    print("  Detalhes dos erros:")
    for e in erros[:5]:
        print(f"    - ID {e['id']}: {e['erro'][:80]}")

---
## Célula 7 — Validação do Dataset: Campos RAGAS Obrigatórios

In [ ]:
# Campos obrigatórios para avaliação RAGAS
CAMPOS_RAGAS = ["question", "answer", "contexts", "ground_truth"]

problemas: list[str] = []

for i, par in enumerate(dataset_completo):
    for campo in CAMPOS_RAGAS:
        # Verifica presença do campo
        if campo not in par:
            problemas.append(f"Par {i}: campo '{campo}' ausente")
        # Verifica se não está vazio
        elif not par[campo]:
            problemas.append(f"Par {i}: campo '{campo}' está vazio")
        # Para 'contexts', verifica se é lista não-vazia
        elif campo == "contexts" and (not isinstance(par[campo], list) or len(par[campo]) == 0):
            problemas.append(f"Par {i}: campo 'contexts' deve ser lista não-vazia")

print("=" * 60)
print("VALIDAÇÃO DO DATASET")
print("=" * 60)
print(f"Total de pares    : {len(dataset_completo)}")
print(f"Campos verificados: {CAMPOS_RAGAS}")

if problemas:
    print(f"\n[ATENÇÃO] {len(problemas)} problema(s) encontrado(s):")
    for p in problemas[:10]:
        print(f"  - {p}")
else:
    print("\n[OK] Todos os pares possuem os 4 campos RAGAS presentes e não-vazios.")

print("\n--- Exemplos de 3 pares completos ---")
for i, par in enumerate(dataset_completo[:3]):
    print(f"\nPar {i+1} — ID: {par.get('id', 'N/A')} | Tipo: {par.get('tipo', 'N/A')} | Dificuldade: {par.get('dificuldade', 'N/A')}")
    print(f"  question    : {par['question'][:120]}")
    print(f"  answer      : {par['answer'][:120]}")
    print(f"  contexts[0] : {par['contexts'][0][:120]}")
    print(f"  ground_truth: {par['ground_truth'][:120]}")

---
## Célula 8 — Exportação do Dataset para JSON e CSV

In [ ]:
import os as os_module

# Define caminhos de saída
CAMINHO_JSON = "dataset_avaliacao_completo.json"
CAMINHO_CSV  = "dataset_avaliacao_completo.csv"

# Exporta JSON (mantém lista de contextos como array)
with open(CAMINHO_JSON, "w", encoding="utf-8") as f:
    json.dump(dataset_completo, f, ensure_ascii=False, indent=2)

tamanho_json = os_module.path.getsize(CAMINHO_JSON) / 1024
print(f"[OK] JSON exportado: '{CAMINHO_JSON}' — {tamanho_json:.1f} KB")

# Prepara DataFrame para CSV (contextos como string separada por '|||')
df_export = pd.DataFrame(dataset_completo)
df_export["contexts_str"] = df_export["contexts"].apply(
    lambda lst: " ||| ".join(lst) if isinstance(lst, list) else str(lst)
)
colunas_csv = ["id", "tipo", "dificuldade", "question", "answer", "contexts_str", "ground_truth"]
df_export[colunas_csv].to_csv(CAMINHO_CSV, index=False, encoding="utf-8")

tamanho_csv = os_module.path.getsize(CAMINHO_CSV) / 1024
print(f"[OK] CSV exportado : '{CAMINHO_CSV}' — {tamanho_csv:.1f} KB")
print(f"\nArquivos prontos para uso nos LABs subsequentes.")

---
## Célula 9 — Análise Descritiva do Dataset Gerado

In [ ]:
import statistics

# Calcula métricas descritivas
comprimentos_answer  = [len(p["answer"]) for p in dataset_completo]
n_contextos          = [len(p["contexts"]) for p in dataset_completo]
comprimentos_gt      = [len(p["ground_truth"]) for p in dataset_completo]
comprimentos_question = [len(p["question"]) for p in dataset_completo]

print("=" * 60)
print("ANÁLISE DESCRITIVA DO DATASET GERADO")
print("=" * 60)

print("\n--- Comprimento das Respostas Geradas (characters) ---")
print(f"  Média    : {statistics.mean(comprimentos_answer):.0f}")
print(f"  Mediana  : {statistics.median(comprimentos_answer):.0f}")
print(f"  Mínimo   : {min(comprimentos_answer)}")
print(f"  Máximo   : {max(comprimentos_answer)}")

print("\n--- Comprimento das Perguntas (characters) ---")
print(f"  Média    : {statistics.mean(comprimentos_question):.0f}")
print(f"  Mediana  : {statistics.median(comprimentos_question):.0f}")

print("\n--- Número de Contextos por Par ---")
print(f"  Média    : {statistics.mean(n_contextos):.1f}")
print(f"  Mínimo   : {min(n_contextos)}")
print(f"  Máximo   : {max(n_contextos)}")

print("\n--- Comprimento dos Ground Truths (characters) ---")
print(f"  Média    : {statistics.mean(comprimentos_gt):.0f}")
print(f"  Mediana  : {statistics.median(comprimentos_gt):.0f}")

print("\n--- Distribuição por Tipo Jurídico ---")
df_analise = pd.DataFrame(dataset_completo)
if "tipo" in df_analise.columns:
    dist = df_analise["tipo"].value_counts()
    for tipo, cnt in dist.items():
        print(f"  {tipo:<25} : {cnt:>3} pares ({100*cnt/len(dataset_completo):.1f}%)")

print("\n--- Distribuição por Dificuldade ---")
if "dificuldade" in df_analise.columns:
    dist_dif = df_analise["dificuldade"].value_counts()
    for dif, cnt in dist_dif.items():
        print(f"  {dif:<10} : {cnt:>3} pares ({100*cnt/len(dataset_completo):.1f}%)")

---

## Checklist de Conclusão — LAB 1

Revise cada item antes de prosseguir para o **LAB 2**:

| # | Item | Status |
|---|------|--------|
| 1 | Dependências instaladas sem erros críticos | ☐ |
| 2 | Variáveis de ambiente configuradas corretamente | ☐ |
| 3 | Conexão com OpenSearch testada (ou fallback ativado) | ☐ |
| 4 | Conexão com vLLM testada (ou fallback ativado) | ☐ |
| 5 | Encoder BGE-M3 carregado com sucesso | ☐ |
| 6 | Dataset de 50 pares processado com barra de progresso | ☐ |
| 7 | Validação RAGAS: todos os 4 campos presentes e não-vazios | ☐ |
| 8 | Arquivos exportados: `dataset_avaliacao_completo.json` e `.csv` | ☐ |

> **Nota ABNT:** Este laboratório integra o projeto de pesquisa aplicada do MBA, seguindo as diretrizes da ABNT NBR 6022:2018 (Artigo em publicação periódica) e ABNT NBR 14724:2011 (Trabalhos acadêmicos). Os modelos e frameworks utilizados devem ser devidamente referenciados nas publicações derivadas deste trabalho.